# Worker GPU Colab per Flask locale

Questo notebook serve solo a far girare **colab_worker.py** su Colab.

Il server principale **app.py** resta sul Mac. Unity deve chiamare il Mac, non Colab.

Flusso:

```text
Unity/curl -> app.py locale sul Mac -> colab_worker.py su Colab -> modelli GPU
```


In [9]:
# 1) Configurazione
REPO_URL = "https://github.com/niccolociotti/Generazione-e-refinement-di-modelli-3D-da-immagini.git"
PROJECT_DIR = "/content/ProgettoCG"

if not REPO_URL:
    raise ValueError("Inserisci l'URL GitHub del progetto in REPO_URL.")

print("Repo:", REPO_URL)
print("Project dir:", PROJECT_DIR)

Repo: https://github.com/niccolociotti/Generazione-e-refinement-di-modelli-3D-da-immagini.git
Project dir: /content/ProgettoCG


In [10]:
# 2) Clona o aggiorna il progetto su Colab
import os
import subprocess
from pathlib import Path

project_path = Path(PROJECT_DIR)

if project_path.exists():
    os.chdir(project_path)
    subprocess.run(["git", "pull"], check=True)
else:
    os.chdir("/content")
    subprocess.run(["git", "clone", REPO_URL, str(project_path)], check=True)
    os.chdir(project_path)

print("cwd:", os.getcwd())
subprocess.run(["ls", "-la"], check=True)


Cloning into 'ProgettoCG'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: '/content/ProgettoCG'

In [ ]:
# 3) Installa ComfyUI e dipendenze per Colab
import os
from pathlib import Path

os.chdir(PROJECT_DIR)

if not Path("ComfyUI").exists():
    !git clone https://github.com/comfyanonymous/ComfyUI
else:
    print("ComfyUI gia' presente")

!python -m pip install -q -r ComfyUI/requirements.txt
!python -m pip install -q -r requirements-colab.txt

print("Dipendenze installate")

In [ ]:
# 4) Scarica i modelli su Colab
# Nota: sono circa 14.5 GB totali. Questa cella puo' metterci parecchio.
import os

os.chdir(PROJECT_DIR)
!mkdir -p ComfyUI/models/diffusion_models ComfyUI/models/clip ComfyUI/models/vae

!wget -c -O ComfyUI/models/diffusion_models/z-image-turbo-fp8-e4m3fn.safetensors "https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors"
!wget -c -O ComfyUI/models/clip/qwen_3_4b.safetensors "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"
!wget -c -O ComfyUI/models/vae/ae.safetensors "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

!ls -lh ComfyUI/models/diffusion_models ComfyUI/models/clip ComfyUI/models/vae

In [ ]:
# 5) Avvia SOLO il worker GPU su Colab, non app.py
import os
import subprocess
import time

os.chdir(PROJECT_DIR)

env = os.environ.copy()
env["COMFYUI_PATH"] = f"{PROJECT_DIR}/ComfyUI"
env["WORKER_PORT"] = "5001"

try:
    WORKER_PROC.terminate()
    time.sleep(1)
except NameError:
    pass

log = open("worker.log", "w")
WORKER_PROC = subprocess.Popen(
    ["python", "colab_worker.py"],
    cwd=PROJECT_DIR,
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
)

time.sleep(8)
print("Worker PID:", WORKER_PROC.pid)
!tail -n 80 worker.log

In [ ]:
# 6) Esponi il worker Colab con cloudflared
import os
import re
import subprocess
import time
from pathlib import Path

os.chdir(PROJECT_DIR)

if not Path("cloudflared").exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

try:
    CLOUDFLARED_PROC.terminate()
    time.sleep(1)
except NameError:
    pass

CLOUDFLARED_PROC = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = CLOUDFLARED_PROC.stdout.readline()
    if not line:
        continue
    print(line, end="")
    match = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    raise RuntimeError("URL cloudflared non trovato. Controlla output e riesegui la cella.")

print("\nCOPIA QUESTO SUL MAC:")
print(f'export REMOTE_IMAGE_WORKER_URL="{public_url}"')
print("\nHealth worker:", public_url + "/health")

In [ ]:
# 7) Test worker Colab dal notebook
import requests

r = requests.get(public_url + "/health", timeout=30)
print(r.status_code)
print(r.text)

# Se models_loaded e' false, guarda worker.log:
!tail -n 80 worker.log